# Pipeline KYC — Inventaire des documents & extraction du justificatif d'identité

**Objectif de ce notebook**

1. Dézipper `kyc_documents.zip` et repérer, dans chaque dossier client, les 5 documents utiles parmi tous
   ceux présents : `JUSTIFICATIF IDENTITE.PDF`, `JUSTIFICATIF DOMICILE.PDF`, `CONVENTION COMPTE.PDF`,
   `FATCA.PDF`, `CARTON SIGNATURE.PDF`.
2. Produire un **rapport d'inventaire** (CSV) indiquant, pour chaque client, quels documents sont présents.
3. Déterminer la **liste des clients cibles** : ceux pour lesquels `JUSTIFICATIF IDENTITE.PDF` existe.
4. Pour ces clients, **extraire automatiquement l'information** du `JUSTIFICATIF IDENTITE.PDF` (document
   scanné, potentiellement multi-page, multilingue, mal orienté ou de mauvaise qualité) à l'aide du modèle
   **Qwen3.5** (vision-langage) hébergé localement sur votre ModelHub Domino — aucune donnée ne sort de
   votre environnement.

**Hypothèses et choix retenus** (tout est regroupé en Partie 1 pour être ajusté facilement) :

- La liste des fichiers cibles contient `CARTON SIGNATURE.PDF` : l'énoncé initial mentionnait
  *"CARTON SIGNATUTE.PDF"*, probablement une coquille — corrigez `TARGET_FILES` si le nom réel diffère
  dans vos dossiers.
- Le `config.json` fourni indique `"model_type": "qwen3_5"`, c'est-à-dire l'architecture **Qwen3.5**
  (multimodale texte/image/vidéo). Ce nom diffère légèrement de celui du dossier ModelHub ("Qwen3.8"),
  probablement une convention de nommage interne : le code se base sur l'architecture réellement déclarée
  dans le config.json, pas sur le nom du dossier.
- La comparaison des noms de fichiers est **insensible à la casse** (les scans bancaires ont rarement une
  casse homogène), et chaque dossier client est parcouru **récursivement**.
- Seul `JUSTIFICATIF IDENTITE.PDF` est traité par le modèle dans ce notebook, comme demandé. Les 4 autres
  types de documents sont inventoriés ; le même patron de code (Parties 4 à 8) pourra leur être appliqué
  ensuite.
- Le traitement par lot est **reprenable** : chaque client traité produit un fichier JSON individuel, donc
  une interruption (crash, timeout) ne fait pas perdre le travail déjà effectué.
- Pour la conversion PDF → image, on utilise **PyMuPDF** plutôt que `pdf2image`/Poppler : aucune dépendance
  système (binaire externe) à installer, ce qui est plus robuste dans un environnement managé comme Domino.
- Ce notebook ne fige pas de numéros de version exacts pour `torch`/`transformers` (écosystème qui évolue
  vite) : il installe une **version plancher connue pour fonctionner avec Qwen3.5**, puis **enregistre les
  versions réellement installées** dans un fichier (`environnement_installe.txt`) pour la traçabilité.
- Le modèle est chargé via la classe générique **`AutoModelForImageTextToText`** (et non la classe
  spécifique `Qwen3_5ForConditionalGeneration`) : c'est l'approche que vous avez adoptée, et c'est aussi
  celle recommandée par la fiche officielle du modèle sur Hugging Face — elle résout automatiquement la
  bonne architecture à partir de `config.json`, sans dépendre d'un nom de classe figé dans ce notebook.
- Le prétraitement (Partie 4/5) est **systématique et dans un ordre précis** — pas une case à cocher au
  cas par cas — car en réalité, les scans bancaires sont mal orientés ET de mauvaise qualité, pas l'un ou
  l'autre : chaque page est d'abord ramenée à une résolution sûre pour le modèle, puis son contraste est
  amélioré, puis sa rotation franche corrigée, et enfin son inclinaison résiduelle corrigée en dernier.
- Un plafond de résolution (`MAX_IMAGE_DIMENSION_MODEL`, Partie 1) est appliqué **avant tout appel au
  modèle** : des images à très haute résolution (un rendu PDF à 300 DPI les dépasse largement) ont été
  signalées par la communauté Hugging Face comme provoquant une réponse dégénérée de Qwen3.5 sur les
  entrées image (une suite de caractères répétés, ex. `!!!!!!!!!!!!!!!`) — **y compris sur des documents
  par ailleurs bien orientés et lisibles**. C'est très probablement la cause de la réponse observée lors
  de votre test (voir la checklist de dépannage, Partie 5bis).
- Un diagnostic de répartition GPU/CPU du modèle est affiché juste après son chargement (Partie 5) : un
  déchargement partiel sur CPU, même minime, est généralement le facteur le plus déterminant pour la
  lenteur d'un traitement par lot (Partie 7) — bien avant les autres optimisations apportées.
- `TRUST_REMOTE_CODE` (Partie 1, par défaut `True`) autorise le téléchargement du noyau de calcul FP8
  optimisé (RedHatAI, via la librairie `kernels`) plutôt que de basculer silencieusement sur une
  implémentation de repli qui a fait planter CUDA lors de votre test (`AcceleratorError` dans
  `w8a8_block_fp8_matmul`) — voir l'explication complète en Partie 5 et l'alternative bf16 si votre
  politique de sécurité impose de le désactiver.

## Partie 0 — Installation des librairies

Installez dans cet ordre : d'abord les librairies de fichiers/images (légères, sans risque), puis la pile
modèle (`transformers`, `accelerate`, `compressed-tensors`), et enfin `torch` — à adapter impérativement à
la version CUDA de votre environnement Domino (voir commentaire ci-dessous). Si `torch` est déjà préinstallé
dans votre image Domino (fréquent sur les environnements GPU), vous pouvez sauter cette ligne.

In [ ]:
# --- Traitement de fichiers / PDF / images (aucune dépendance système requise) ---
%pip install -q "pymupdf>=1.26.0"                   # rendu des pages PDF en images ; respecte la rotation déclarée dans le PDF
%pip install -q "pillow>=10.4.0"                    # manipulation d'images
%pip install -q "opencv-python-headless>=4.10.0"    # redressement (deskew) + contraste ; "headless" = pas de dépendance GUI/libGL
%pip install -q "numpy>=1.26.0"
%pip install -q "pandas>=2.2.0"                     # rapport d'inventaire, consolidation des résultats
%pip install -q "tqdm>=4.66.0"                      # barre de progression du traitement par lot

# --- Pile modèle : Qwen3.5 (vision-langage), quantifié FP8 ---
%pip install -q "transformers>=5.8.0"               # le model_type "qwen3_5" est supporté à partir de la 5.8 ; privilégiez la dernière version stable
%pip install -q "accelerate>=0.34.0"                # requis pour device_map="auto" (répartition automatique sur GPU)
%pip install -q "compressed-tensors>=0.7.0"         # requis pour décoder les poids quantifiés FP8 du checkpoint (cf. quantization_config, Partie 5)

# --- PyTorch : à adapter à VOTRE version CUDA (vérifiez avec `!nvidia-smi`) ---
# Décommentez et ajustez l'URL d'index si torch n'est pas déjà présent dans votre environnement Domino, par ex. :
# %pip install -q torch --index-url https://download.pytorch.org/whl/cu124

# --- Optionnel : accélère l'attention hybride (Gated DeltaNet) de Qwen3.5 ---
# Sans ces paquets, le modèle fonctionne normalement mais bascule automatiquement sur un mode de repli
# PyTorch plus lent pour SES COUCHES D'ATTENTION LINÉAIRE spécifiquement. Leur compilation nécessite un
# toolchain CUDA correspondant exactement à votre torch : à tenter seulement si la vitesse d'inférence
# pose problème, sinon inutile de les installer.
# (Les couches d'attention "classique" du modèle utilisent déjà le backend natif et rapide de PyTorch —
# attn_implementation="sdpa", activé automatiquement en Partie 5 — qui ne nécessite aucun paquet en plus.)
# %pip install -q -U kernels
# %pip install -q causal-conv1d --no-build-isolation

## Imports groupés

Toutes les librairies utilisées dans ce notebook, importées une seule fois ici.

In [ ]:
# Bibliothèque standard
import os
import re
import io
import sys
import json
import time
import shutil
import zipfile
import logging
import platform
import subprocess
from pathlib import Path
from datetime import datetime

# Traitement de données / fichiers / images
import numpy as np
import pandas as pd
import cv2
import pymupdf
from PIL import Image
from tqdm.auto import tqdm

# Modèle
import torch
import transformers
from transformers import AutoProcessor, AutoModelForImageTextToText

print("Toutes les librairies ont été importées avec succès.")

## Partie 1 — Configuration

Tous les paramètres modifiables du pipeline sont centralisés ici : chemins, liste des fichiers cibles,
options de prétraitement d'image. C'est le seul endroit à modifier pour adapter le notebook à votre
environnement exact.

In [ ]:
# ============================== CHEMINS ==============================
ZIP_PATH = Path("kyc_documents.zip")                       # <-- à adapter : emplacement réel du zip dans Domino
WORK_DIR = Path("kyc_pipeline_workdir")                     # tous les fichiers produits par ce notebook y seront rangés

RAW_EXTRACT_DIR  = WORK_DIR / "01_extraction_brute"          # dézippage complet et brut
FILTERED_DIR     = WORK_DIR / "02_documents_cibles"           # uniquement les 5 fichiers utiles, par client
RESULTS_DIR      = WORK_DIR / "03_resultats_identite"         # un fichier JSON par client traité (reprenable)

REPORT_PATH            = WORK_DIR / "rapport_inventaire_kyc.csv"
TARGET_CLIENTS_PATH    = WORK_DIR / "liste_clients_cibles.csv"
COMBINED_RESULTS_JSON  = WORK_DIR / "resultats_extraction_identite.json"
COMBINED_RESULTS_CSV   = WORK_DIR / "resultats_extraction_identite.csv"
LOG_PATH               = WORK_DIR / "pipeline_kyc.log"
ENV_SNAPSHOT_PATH      = WORK_DIR / "environnement_installe.txt"

MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"  # chemin fourni

# ============================== FICHIERS CIBLES ==============================
# Comparaison insensible à la casse (voir normalize() en Partie 2) : la casse ci-dessous n'a pas besoin de
# correspondre exactement à celle des fichiers réels sur le disque.
TARGET_FILES = [
    "JUSTIFICATIF IDENTITE.PDF",
    "JUSTIFICATIF DOMICILE.PDF",
    "CONVENTION COMPTE.PDF",
    "FATCA.PDF",
    "CARTON SIGNATURE.PDF",     # "SIGNATURE" ; remplacez par "SIGNATUTE" si c'est réellement ce nom-là chez vous
]
IDENTITY_FILE = "JUSTIFICATIF IDENTITE.PDF"    # doit être un élément exact de TARGET_FILES ci-dessus

# ============================== PARAMÈTRES DE TRAITEMENT ==============================
PDF_RENDER_DPI               = 300     # résolution de rendu PDF -> image. resize_for_model (Partie 4)
                                        # réduit ensuite cette image avant l'envoi au modèle (voir
                                        # MAX_IMAGE_DIMENSION_MODEL) : cette valeur n'affecte donc que la
                                        # qualité de l'image intermédiaire et le temps de rendu PDF, pas la
                                        # résolution réellement vue par le modèle.
MAX_IMAGE_DIMENSION_MODEL    = 1568    # plafond de résolution (plus grand côté, en pixels) AVANT tout appel
                                        # au modèle. Ce n'est PAS une option de confort : des images plus
                                        # grandes ont été signalées par la communauté Hugging Face comme
                                        # provoquant une réponse dégénérée de Qwen3.5 sur les entrées image
                                        # (voir resize_for_model, Partie 4). Diminuez cette valeur (ex. 1024
                                        # ou 768) si le problème persiste malgré tout — voir la checklist de
                                        # dépannage en Partie 5bis.
ENABLE_ORIENTATION_CHECK     = True    # corrige les rotations franches (90/180/270°) via le modèle
ENABLE_SKEW_CORRECTION       = True    # corrige les légères inclinaisons (quelques degrés) via OpenCV
ENABLE_CONTRAST_ENHANCEMENT  = True    # améliore le contraste des scans de mauvaise qualité (CLAHE)
MAX_NEW_TOKENS_EXTRACTION    = 2048    # longueur max. de la réponse du modèle pour l'extraction structurée
FORCE_REPROCESS              = False   # True = retraite même les clients déjà traités (sinon reprise automatique)

# ============================== SÉCURITÉ / CODE DISTANT (noyaux de calcul FP8) ==============================
# Le chargement d'un checkpoint quantifié FP8 (compressed-tensors) peut tenter de télécharger et D'EXÉCUTER un
# noyau de calcul optimisé (kernel CUTLASS) depuis un dépôt Hugging Face tiers (ex. "RedHatAI/quantization"),
# via la librairie `kernels`. Depuis son passage aux "éditeurs de confiance", `kernels` n'autorise PAR DÉFAUT
# que les dépôts explicitement approuvés par Hugging Face ; sinon, le chargement du kernel échoue avec un
# avertissement ("could not verify publisher trust status") et bascule silencieusement vers une implémentation
# de repli (Triton) plus lente ET, dans certains environnements, instable au point de faire planter CUDA en
# cours de génération (voir la checklist de dépannage, Partie 5bis, si vous rencontrez une erreur
# "AcceleratorError"/"cudaErrorUnknown").
# TRUST_REMOTE_CODE=True autorise ce téléchargement/exécution de code tiers — une VRAIE décision de sécurité,
# pas un simple réglage de confort : à faire valider par votre équipe sécurité/infra avant un déploiement en
# production dans un contexte bancaire, même si l'éditeur concerné ici (RedHatAI, l'organisation Red Hat) est
# une source réputée légitime. Passez à False si votre politique ne permet pas cette exécution de code distant
# — voir l'alternative sans noyau optimisé (chargement en bf16) proposée en Partie 5 dans ce cas.
TRUST_REMOTE_CODE = True

# ============================== INITIALISATION ==============================
for d in (WORK_DIR, RAW_EXTRACT_DIR, FILTERED_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
    force=True,  # évite les handlers dupliqués si la cellule est réexécutée
)
logger = logging.getLogger("kyc_pipeline")
logger.info("Configuration chargée. Répertoire de travail : %s", WORK_DIR.resolve())

# Remarque « protection des données » : les logs ne contiennent volontairement que des identifiants client
# et des statuts techniques — jamais les données personnelles extraites elles-mêmes.

Vérification de l'environnement (versions installées, GPU disponible) et sauvegarde d'un instantané des
versions réellement présentes, pour la traçabilité / l'audit (utile en contexte bancaire réglementé).

In [ ]:
print(f"Date d'exécution        : {datetime.now().isoformat(timespec='seconds')}")
print(f"Python                  : {sys.version.split()[0]} ({platform.system()} {platform.release()})")
print(f"PyTorch                 : {torch.__version__}")
print(f"Transformers            : {transformers.__version__}")
print(f"CUDA disponible         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i} : {props.name} — {props.total_memory / 1e9:.1f} Go")
else:
    print("Aucun GPU détecté. L'inférence sur un modèle de cette taille sera très lente, voire impraticable, sur CPU.")

with open(ENV_SNAPSHOT_PATH, "w", encoding="utf-8") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
logger.info("Snapshot de l'environnement sauvegardé -> %s", ENV_SNAPSHOT_PATH)

## Partie 2 — Dézippage et inventaire des documents KYC

On dézippe `kyc_documents.zip`, puis pour **chaque dossier client**, on recherche les 5 fichiers cibles
(recherche récursive, insensible à la casse). Les fichiers trouvés sont copiés dans une arborescence propre
(`FILTERED_DIR/<client_id>/<nom_canonique>.PDF`), et leur présence/absence est consignée dans un tableau
qui sera sauvegardé en CSV.

In [ ]:
def normalize(name: str) -> str:
    """Normalise un nom de fichier pour une comparaison insensible à la casse et aux espaces superflus."""
    return name.strip().upper()

TARGET_FILES_NORM = {normalize(f): f for f in TARGET_FILES}

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Archive introuvable : {ZIP_PATH.resolve()}. Vérifiez ZIP_PATH dans la cellule de configuration (Partie 1)."
    )

if RAW_EXTRACT_DIR.exists():
    shutil.rmtree(RAW_EXTRACT_DIR)
RAW_EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(RAW_EXTRACT_DIR)
logger.info("Archive dézippée -> %s", RAW_EXTRACT_DIR.resolve())

# Le zip peut soit contenir directement les dossiers clients, soit un unique dossier racine qui les englobe :
# on détecte automatiquement le bon niveau.
entries = list(RAW_EXTRACT_DIR.iterdir())
DATA_ROOT = entries[0] if (len(entries) == 1 and entries[0].is_dir()) else RAW_EXTRACT_DIR

client_folders = sorted(p for p in DATA_ROOT.iterdir() if p.is_dir())
logger.info("%d dossier(s) client détecté(s) sous %s", len(client_folders), DATA_ROOT)

In [ ]:
rows = []
for client_dir in tqdm(client_folders, desc="Inventaire des documents"):
    client_id = client_dir.name
    all_files = [p for p in client_dir.rglob("*") if p.is_file()]   # recherche récursive

    found = {}  # nom_cible_canonique -> chemin réel trouvé
    for f in all_files:
        canonical = TARGET_FILES_NORM.get(normalize(f.name))
        if canonical:
            found[canonical] = f

    row = {"client_id": client_id}
    for target in TARGET_FILES:
        row[target] = target in found
    row["nb_documents_cibles_trouves"] = len(found)
    row["dossier_complet"] = len(found) == len(TARGET_FILES)
    rows.append(row)

    if found:
        dest_dir = FILTERED_DIR / client_id
        dest_dir.mkdir(parents=True, exist_ok=True)
        for canonical_name, src_path in found.items():
            shutil.copy2(src_path, dest_dir / canonical_name)

inventory_df = pd.DataFrame(rows).set_index("client_id").sort_index()
inventory_df.to_csv(REPORT_PATH, encoding="utf-8-sig")

logger.info("Rapport d'inventaire sauvegardé -> %s", REPORT_PATH.resolve())
print(f"\n{len(inventory_df)} client(s) au total.")
for target in TARGET_FILES:
    print(f"  - {target:<32} présent chez {int(inventory_df[target].sum())} client(s)")
print(f"  - Dossiers complets (5/5)         : {int(inventory_df['dossier_complet'].sum())}")

inventory_df

## Partie 3 — Liste des clients cibles

Les **clients cibles** sont ceux pour lesquels `JUSTIFICATIF IDENTITE.PDF` a été trouvé : ce sont eux qui
seront traités dans la suite du notebook.

In [ ]:
target_clients = inventory_df.index[inventory_df[IDENTITY_FILE]].tolist()

pd.Series(target_clients, name="client_id").to_csv(TARGET_CLIENTS_PATH, index=False, encoding="utf-8-sig")
logger.info("%d client(s) cible(s) -> %s", len(target_clients), TARGET_CLIENTS_PATH.resolve())

print(f"{len(target_clients)} client(s) cible(s) sur {len(inventory_df)} :")
print(target_clients)

## Partie 4 — Prétraitement des scans

Dans la réalité, un scan bancaire n'est pas juste « un peu incliné » OU « de moins bonne qualité » : les
deux problèmes se cumulent, et s'y ajoute parfois une **rotation franche** (document scanné à l'envers ou
sur le côté). Le prétraitement ci-dessous n'est donc pas une liste d'options à activer au cas par cas,
mais une chaîne **systématique**, appliquée à chaque page **dans un ordre précis** (assemblée dans
`preprocess_page`, Partie 5, une fois le modèle chargé) :

1. **`pdf_to_images`** : convertit chaque page du PDF en image haute résolution. PyMuPDF applique déjà
   automatiquement la rotation éventuellement déclarée dans les métadonnées de la page (`/Rotate`).
2. **`resize_for_model`** *(nouveau)* : plafonne la résolution avant tout envoi au modèle. Appliquée en
   tout premier — voir sa docstring ci-dessous : ce n'est pas une question de vitesse uniquement, une
   image trop grande peut faire répondre le modèle n'importe quoi.
3. **`enhance_image`** : améliore le contraste (CLAHE) et la netteté d'un scan de mauvaise qualité.
   Appliquée tôt, avant toute décision géométrique ou basée sur le modèle : un scan plus net et mieux
   contrasté aide À LA FOIS la détection d'orientation (étape 4, faite par le modèle) et la détection
   géométrique d'inclinaison (étape 5, faite par OpenCV).
4. **`detect_and_fix_orientation`** *(Partie 5)* : corrige une **rotation franche** (90°/180°/270°) via
   le modèle — non détectable de façon fiable par la seule géométrie, c'est lui qui identifie le mieux
   le sens de lecture réel du texte (y compris en écriture manuscrite ou en arabe).
5. **`correct_skew`** : corrige une **légère inclinaison résiduelle** (quelques degrés), typique d'un
   document mal aligné sur le scanner, via une détection géométrique (OpenCV). Appliquée **en dernier**,
   une fois le document déjà globalement à l'endroit (étape 4) : sa détection d'angle suppose un texte à
   peu près horizontal, une hypothèse qui ne tient pas tant qu'une rotation franche n'a pas été corrigée.

In [ ]:
def pdf_to_images(pdf_path: Path, dpi: int = PDF_RENDER_DPI) -> list:
    """Convertit chaque page d'un PDF (y compris scanné) en une image PIL RGB haute résolution."""
    images = []
    with pymupdf.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(dpi=dpi)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            images.append(img)
    return images

In [ ]:
def resize_for_model(image: Image.Image, max_dimension: int = MAX_IMAGE_DIMENSION_MODEL) -> Image.Image:
    """Limite la plus grande dimension de l'image à max_dimension avant tout envoi au modèle.

    Ce n'est PAS une option de confort : des images de très haute résolution (un rendu PDF à 300 DPI
    dépasse largement max_dimension) ont été signalées par la communauté Hugging Face comme provoquant
    une réponse DÉGÉNÉRÉE du modèle Qwen3.5 sur les entrées image -- une suite de caractères répétés,
    ex. "!!!!!!!!!!!!!!!", au lieu d'une vraie réponse -- y compris sur des images par ailleurs bien
    orientées et parfaitement lisibles à l'oeil nu. Réduire la résolution avant l'appel au modèle a
    résolu le problème dans les cas rapportés. Cette étape accélère aussi nettement l'inférence, la
    résolution de l'image étant directement liée au nombre de "tokens image" à traiter par le modèle.
    Ne redimensionne jamais vers le HAUT (une petite image reste inchangée).
    """
    w, h = image.size
    longest_side = max(w, h)
    if longest_side <= max_dimension:
        return image
    scale = max_dimension / longest_side
    new_size = (max(1, round(w * scale)), max(1, round(h * scale)))
    return image.resize(new_size, Image.LANCZOS)

In [ ]:
def correct_skew(image: Image.Image, max_angle: float = 15.0) -> Image.Image:
    """Corrige une légère inclinaison (quelques degrés) via une détection géométrique du contenu texte.
    Ignore volontairement les angles > max_angle : au-delà, il s'agit probablement d'une rotation franche
    (90/180/270°), gérée séparément par detect_and_fix_orientation (Partie 5)."""
    arr = np.array(image)
    gray = cv2.bitwise_not(cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY))
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 50:
        return image  # page quasi blanche : rien à corriger

    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) > max_angle or abs(angle) < 0.1:
        return image

    h, w = arr.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(arr, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(rotated)

In [ ]:
def enhance_image(image: Image.Image) -> Image.Image:
    """Améliore un scan de mauvaise qualité : contraste local (CLAHE sur le canal de luminance), puis
    léger renforcement de netteté (utile pour les scans flous). Volontairement PAS de binarisation ni de
    débruitage agressif : cela risquerait de dégrader la photo d'identité et les éléments colorés du
    document. Le résultat est explicitement re-borné à [0, 255] en uint8 avant conversion en image : un
    tableau hors de cette plage (dépassement possible après le renforcement de netteté) donnerait une
    image corrompue une fois envoyée au modèle."""
    arr = np.array(image)
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    result = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

    # Renforcement léger de netteté (unsharp mask) : aide les scans flous sans créer d'artefacts visibles
    # si le scan était déjà net.
    blurred = cv2.GaussianBlur(result, (0, 0), sigmaX=2.0)
    sharpened = cv2.addWeighted(result, 1.5, blurred, -0.5, 0)
    sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
    return Image.fromarray(sharpened)

## Partie 5 — Chargement du modèle Qwen3.5 (vision-langage) en local

Le `config.json` fourni indique `"model_type": "qwen3_5"` : il s'agit de l'architecture **Qwen3.5**, un
modèle nativement multimodal (texte / image / vidéo), supporté par `transformers` à partir de la version
5.8. Le checkpoint est quantifié **FP8** (probablement au format `compressed-tensors`, d'où sa dépendance
installée en Partie 0) : la cellule suivante inspecte le `quantization_config` réel pour confirmer le
format avant chargement.

On charge le modèle via **`AutoModelForImageTextToText`** (résout automatiquement la classe
`Qwen3_5ForConditionalGeneration` à partir de `config.json` — approche recommandée par la fiche officielle
du modèle plutôt que d'importer la classe spécifique en dur), avec `dtype="auto"` (respecte la précision
déjà présente dans le checkpoint, donc le FP8) et `device_map="auto"` (répartition automatique sur le(s)
GPU disponibles). On tente `attn_implementation="sdpa"` (backend d'attention natif PyTorch, rapide, sans
dépendance supplémentaire) avec repli automatique si votre environnement ne le supporte pas.

**Important pour un checkpoint FP8** : `trust_remote_code` (`TRUST_REMOTE_CODE`, Partie 1) conditionne le
téléchargement du noyau de calcul optimisé pour le FP8 — sans lui, transformers bascule sur une
implémentation de repli qui peut être instable (voir Partie 5bis en cas d'erreur CUDA).

In [ ]:
config_path = Path(MODEL_PATH) / "config.json"
if not config_path.exists():
    raise FileNotFoundError(f"config.json introuvable à {config_path} — vérifiez MODEL_PATH (Partie 1).")

with open(config_path, encoding="utf-8") as f:
    model_config = json.load(f)

print("model_type            :", model_config.get("model_type"))
print("transformers_version  :", model_config.get("transformers_version"), "(version utilisée lors de la sauvegarde du modèle)")
print("quantization_config   :")
print(json.dumps(model_config.get("quantization_config", {}), indent=2, ensure_ascii=False))

quant_method = str(model_config.get("quantization_config", {}).get("quant_method", "")).lower()
if "compressed" in quant_method or "fp8" in quant_method:
    print(
        "\nCheckpoint FP8 (compressed-tensors) détecté. Pour ce type de checkpoint, transformers essaie "
        "d'utiliser un noyau de calcul optimisé (DeepGEMM/CUTLASS, souvent récupéré depuis le Hub via la "
        "librairie `kernels`) ; s'il ne peut pas être chargé (GPU non compatible, ou dépôt non reconnu comme "
        "'éditeur de confiance' -- voir TRUST_REMOTE_CODE, Partie 1), transformers bascule sur une "
        "implémentation Triton plus lente et, sur certains environnements, instable. Si le chargement du "
        "modèle ci-dessous affiche un avertissement 'CUTLASS quantization kernel' / 'publisher trust status', "
        "voir Partie 5bis en cas d'erreur CUDA pendant la génération."
    )

In [ ]:
t0 = time.time()
logger.info("Chargement du modèle depuis %s ...", MODEL_PATH)

# Si cette ligne échoue avec une erreur "unrecognized model type" pour "qwen3_5", votre version de
# transformers est probablement antérieure à la 5.8 : exécutez `%pip install -U transformers` puis
# redémarrez le kernel.
#
# ⚠️ Si vous avez DÉJÀ vu une erreur CUDA (AcceleratorError / cudaErrorUnknown / RuntimeError CUDA...) dans
# cette session, REDÉMARREZ LE KERNEL avant de relancer cette cellule : un contexte CUDA corrompu par une
# erreur ne se répare pas en réexécutant simplement le code, y compris avec les correctifs ci-dessous.
#
# attn_implementation="sdpa" accélère les couches d'attention "classique" (backend natif PyTorch, aucune
# dépendance supplémentaire). On retente sans ce réglage si votre environnement ne le supporte pas pour
# cette architecture, plutôt que de faire échouer tout le chargement pour un gain de vitesse secondaire.
#
# trust_remote_code=TRUST_REMOTE_CODE (Partie 1) autorise le téléchargement du noyau FP8 optimisé
# (voir la cellule précédente) plutôt que la bascule vers l'implémentation de repli Triton.
try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_PATH,
        dtype="auto",                       # respecte la précision du checkpoint (FP8 via compressed-tensors)
        device_map="auto",                  # répartit automatiquement sur le(s) GPU disponible(s)
        attn_implementation="sdpa",
        trust_remote_code=TRUST_REMOTE_CODE,
    )
except (ValueError, TypeError) as e:
    logger.warning("attn_implementation='sdpa' indisponible (%s) — nouvelle tentative sans ce réglage.", e)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_PATH,
        dtype="auto",
        device_map="auto",
        trust_remote_code=TRUST_REMOTE_CODE,
    )

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=TRUST_REMOTE_CODE)
model.eval()

# --- Alternative si le noyau FP8 optimisé continue de poser problème (TRUST_REMOTE_CODE=False imposé par
# votre politique de sécurité, ou l'erreur CUDA persiste malgré TRUST_REMOTE_CODE=True) : chargez en bf16
# pour contourner ENTIÈREMENT le chemin de calcul FP8 (DeepGEMM et Triton), au prix d'environ 2x la mémoire
# GPU du checkpoint FP8 -- vérifiez le diagnostic de répartition GPU/CPU ci-dessous après avoir basculé.
# Commentez le bloc try/except ci-dessus et décommentez celui-ci à la place (nécessite un redémarrage du
# kernel si vous avez déjà tenté un chargement FP8 dans cette session) :
#
# model = AutoModelForImageTextToText.from_pretrained(
#     MODEL_PATH,
#     dtype=torch.bfloat16,               # force la déquantification -> évite le chemin FP8 (DeepGEMM/Triton)
#     device_map="auto",
#     trust_remote_code=TRUST_REMOTE_CODE,
# )
# processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=TRUST_REMOTE_CODE)
# model.eval()

logger.info("Modèle chargé en %.1fs.", time.time() - t0)
if torch.cuda.is_available():
    print(f"Mémoire GPU allouée après chargement : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

# --- Diagnostic important pour la VITESSE (Partie 7) ---
# Un modèle de cette taille doit tenir ENTIÈREMENT sur GPU. Si device_map="auto" a dû répartir ne serait-ce
# qu'une couche sur le CPU (mémoire GPU insuffisante), l'inférence devient extrêmement lente (facteur
# 10x-100x, largement plus déterminant que tout autre réglage de ce notebook). Vérifiez ci-dessous
# qu'aucun appareil autre que "cuda:N" n'apparaît.
device_map = getattr(model, "hf_device_map", None)
if device_map:
    devices_used = sorted(set(str(d) for d in device_map.values()))
    print("Répartition du modèle sur les appareils :", devices_used)
    if any(("cpu" in d or "disk" in d) for d in devices_used):
        print(
            "\n⚠️  ATTENTION : une partie du modèle est déchargée sur CPU/disque. C'est très probablement "
            "la cause principale d'un traitement extrêmement lent en Partie 7, avant toute autre "
            "optimisation. Pistes : libérer de la mémoire GPU (autres processus), demander un GPU avec "
            "plus de VRAM, ou utiliser une variante du modèle plus petite/plus quantifiée."
        )
else:
    print("Modèle chargé sans device_map détaillé (mono-GPU ou CPU uniquement probable).")

Fonction générique d'appel au modèle, et détection/correction des rotations franches (90/180/270°) —
cette dernière a besoin du modèle chargé ci-dessus, d'où sa place ici plutôt qu'en Partie 4.

In [ ]:
def _ask_model(images: list, prompt: str, max_new_tokens: int) -> str:
    """Appel générique du modèle avec une ou plusieurs images + une consigne texte. Retourne la réponse brute."""
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,   # garde-fou contre les boucles de répétition (ex. "!!!!!!!!!!!!!!!")
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
        )

    trimmed = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated)]
    return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()


def _looks_degenerate(text: str) -> bool:
    """Détecte une réponse manifestement invalide du modèle (vide, ou dominée par un seul caractère
    répété, ex. "!!!!!!!!!!!!!!!") plutôt qu'une vraie réponse à interpréter. Un tel résultat indique
    presque toujours un problème en amont (image trop volumineuse, problème de génération) -- voir la
    checklist de dépannage en Partie 5bis -- plutôt qu'une réponse du modèle qu'il suffirait de reparser."""
    stripped = text.strip()
    if not stripped:
        return True
    if len(stripped) == 1:
        return not stripped.isalnum()  # un seul caractère : valide seulement s'il est alphanumérique
                                        # (ex. "0" est une vraie réponse d'orientation, "!" ne l'est pas)
    most_common_count = max(stripped.count(c) for c in set(stripped))
    return most_common_count / len(stripped) > 0.8


ORIENTATION_PROMPT = (
    "Regarde cette image de document scanné. Le texte est-il actuellement à l'endroit et horizontal ? "
    "Sinon, de combien de degrés faut-il la faire pivoter DANS LE SENS DES AIGUILLES D'UNE MONTRE pour "
    "qu'il le devienne : 90, 180 ou 270 ? Réponds uniquement par un seul chiffre parmi 0, 90, 180, 270 "
    "— aucun autre mot."
)

def detect_and_fix_orientation(image: Image.Image) -> Image.Image:
    """Détecte une rotation franche (90/180/270°) via le modèle, et corrige l'image en conséquence.

    Journalise explicitement (WARNING) toute réponse dégénérée ou non reconnue plutôt que de la traiter
    silencieusement comme "0° - aucune rotation nécessaire" : une réponse dégénérée ici est un signal
    d'alerte général sur le pipeline, pas une simple absence de rotation à corriger."""
    if not ENABLE_ORIENTATION_CHECK:
        return image
    try:
        answer = _ask_model([image], ORIENTATION_PROMPT, max_new_tokens=8)
    except Exception as e:
        logger.warning("Détection d'orientation impossible (%s) — image conservée telle quelle.", e)
        return image

    if _looks_degenerate(answer):
        logger.warning(
            "Réponse dégénérée du modèle pendant la détection d'orientation (%r) — probable symptôme "
            "d'un problème en amont (voir la checklist de dépannage, Partie 5bis), pas un vrai résultat "
            "d'orientation. Image conservée telle quelle.",
            answer[:30],
        )
        return image

    match = re.search(r"\b(90|180|270|0)\b", answer)
    if not match:
        logger.warning(
            "Réponse inattendue du modèle pendant la détection d'orientation (%r) — aucune rotation "
            "reconnue, image conservée telle quelle.",
            answer[:60],
        )
        return image

    rotation_needed = int(match.group(1))
    if rotation_needed == 0:
        return image
    # PIL.Image.rotate() tourne dans le sens ANTIhoraire pour un angle positif -> signe négatif pour un pivot horaire
    return image.rotate(-rotation_needed, expand=True)

`preprocess_page` assemble les cinq étapes de la Partie 4 + `detect_and_fix_orientation` ci-dessus, dans
l'ordre validé, et est utilisée aussi bien par le test de sanité (Partie 5bis) que par le traitement par
lot (Partie 6/7) : l'aperçu affiché en Partie 5bis correspond ainsi EXACTEMENT à ce que le modèle reçoit
en production — les deux ne peuvent plus diverger silencieusement.

In [ ]:
def preprocess_page(image: Image.Image) -> Image.Image:
    """Chaîne de prétraitement complète d'une page, dans l'ordre validé (voir Partie 4 pour la
    justification détaillée de cet ordre précis) :
    1. resize_for_model            — sécurité/vitesse, en tout premier
    2. enhance_image                — contraste + netteté, avant toute décision géométrique ou modèle
    3. detect_and_fix_orientation   — rotation franche (90/180/270°), via le modèle
    4. correct_skew                 — inclinaison résiduelle, une fois le document déjà à l'endroit
    """
    img = resize_for_model(image)
    img = enhance_image(img) if ENABLE_CONTRAST_ENHANCEMENT else img
    img = detect_and_fix_orientation(img) if ENABLE_ORIENTATION_CHECK else img
    img = correct_skew(img) if ENABLE_SKEW_CORRECTION else img
    return img

## Partie 5bis — Test de sanité

Avant de lancer le traitement complet, on vérifie sur **un seul client** que le modèle charge
correctement, « voit » l'image, et produit une réponse cohérente. Cela évite de découvrir un problème
(chemin, prompt, mémoire GPU...) seulement après avoir attendu la fin d'un traitement par lot de plusieurs
dizaines de minutes. L'image affichée ci-dessous passe par `preprocess_page` — exactement la même chaîne
que celle utilisée en Partie 7 — donc ce que vous voyez ici est bien ce que le modèle reçoit en production.

In [ ]:
if not target_clients:
    print("Aucun client cible — vérifiez le rapport d'inventaire (Partie 2) avant de continuer.")
else:
    sample_client = target_clients[0]
    sample_pages = pdf_to_images(FILTERED_DIR / sample_client / IDENTITY_FILE)
    print(f"Client de test : {sample_client} — {len(sample_pages)} page(s) détectée(s).")

    sample_page = preprocess_page(sample_pages[0])
    display(sample_page)  # aperçu de la page telle que le modèle va la recevoir

    quick_answer = _ask_model(
        [sample_page],
        "En une phrase : quel type de document est visible sur cette image, et le texte te semble-t-il "
        "net et bien orienté ?",
        max_new_tokens=100,
    )
    print("\nRéponse du modèle :", quick_answer)

    if _looks_degenerate(quick_answer):
        print(
            "\n⚠️  Cette réponse ressemble à un résultat DÉGÉNÉRÉ (caractère répété) plutôt qu'à une "
            "vraie réponse. Ce n'est probablement PAS un problème d'orientation ou de qualité de scan : "
            "voir la checklist de dépannage juste en dessous, AVANT de lancer le traitement complet "
            "(Partie 7)."
        )
    else:
        print("\nSi cette réponse est cohérente, vous pouvez passer à la suite en toute confiance.")

### Si la réponse ci-dessus est illisible (ex. une suite de `!`), ou si une erreur CUDA apparaît

**D'abord, une règle absolue : après toute erreur CUDA (`AcceleratorError`, `cudaErrorUnknown`,
`RuntimeError` mentionnant CUDA...), REDÉMARREZ LE KERNEL avant de retenter quoi que ce soit.** Une fois
le contexte CUDA corrompu par une erreur, tous les appels GPU suivants échouent de la même façon dans le
même processus — réexécuter une cellule, même corrigée, ne suffit pas.

**Cas 1 — une vraie erreur Python/CUDA apparaît** (`AcceleratorError`, trace passant par
`w8a8_block_fp8_matmul` / `finegrained_fp8.py`, souvent précédée d'un avertissement "Failed to load
CUTLASS quantization kernel" / "could not verify publisher trust status") : c'est le noyau de calcul FP8
optimisé qui n'a pas pu être chargé (voir Partie 1, `TRUST_REMOTE_CODE`, et l'explication juste après
l'inspection du `quantization_config` en Partie 5), forçant une implémentation de repli qui plante sur
cet environnement. Dans l'ordre, **après avoir redémarré le kernel** :
1. Vérifiez que `TRUST_REMOTE_CODE = True` (Partie 1), puis relancez tout depuis le début du notebook.
2. Si votre politique de sécurité impose `TRUST_REMOTE_CODE = False`, ou si l'erreur persiste malgré
   `True`, basculez sur le chargement bf16 (bloc alternatif en commentaire, Partie 5) qui contourne
   entièrement le calcul FP8 — au prix d'environ 2x la mémoire GPU du checkpoint FP8.
3. Vérifiez `pip show kernels compressed-tensors transformers` : ce sont des librairies très récentes,
   une mise à jour peut suffire.

**Cas 2 — pas d'erreur Python, juste une réponse dégénérée** (suite de caractères répétés, mais aucune
trace CUDA) : ce symptôme a aussi été signalé par plusieurs utilisateurs de Qwen3.5 sur des entrées image
à très haute résolution (voir par exemple la discussion #12 sur la fiche Hugging Face de
`Qwen/Qwen3.5-9B`) — c'est ce que corrige déjà `resize_for_model` (Partie 4), appliqué ci-dessus via
`preprocess_page`. Si ce cas se présente malgré tout : réduisez encore `MAX_IMAGE_DIMENSION_MODEL`
(Partie 1, ex. 1024 puis 768), et vérifiez le diagnostic de répartition GPU/CPU (Partie 5).

Cette checklist est fondée sur des rapports communautaires (Hugging Face) et sur les traces d'erreur que
vous avez rencontrées, pas sur une documentation officielle Qwen/Alibaba — à ajuster si vous identifiez
une autre cause dans votre environnement.

## Partie 6 — Extraction structurée depuis `JUSTIFICATIF IDENTITE.PDF`

Toutes les pages du document (recto/verso, etc.) sont envoyées **ensemble** en une seule requête au
modèle, qui peut ainsi croiser les informations entre pages. Le prompt :

- couvre les documents multilingues (français / arabe / anglais / tamazight / mélange), fréquents sur les
  pièces d'identité algériennes souvent bilingues ;
- demande une sortie **JSON strict**, avec un champ `texte_brut_ocr` (transcription complète, en
  repli/traçabilité) et un champ `champs_incertains` (liste des champs à faible confiance, pour cibler la
  revue manuelle) ;
- interdit explicitement au modèle de traduire les noms propres ou d'inventer une valeur absente.

Adaptez librement le schéma ci-dessous aux champs réellement exigés par votre équipe conformité.

In [ ]:
EXTRACTION_PROMPT = """Tu es un système expert en extraction de données à partir de documents d'identité (carte nationale d'identité, passeport, permis de conduire, titre de séjour, etc.) dans un contexte bancaire de connaissance client (KYC).

Les images fournies sont TOUTES les pages d'un même document, appartenant à un même client (par exemple le recto et le verso d'une carte d'identité). Le document peut être rédigé en français, en arabe, en anglais, en tamazight, ou dans un mélange de ces langues (les cartes d'identité algériennes sont souvent bilingues arabe/français). Le scan peut être de mauvaise qualité, incliné, flou, ou comporter de l'écriture manuscrite.

Analyse l'ensemble des pages fournies et réponds UNIQUEMENT avec un objet JSON valide (sans texte avant ou après, sans balises markdown), respectant exactement ce schéma :

{
  "type_document": string ou null,
  "nom": string ou null,
  "prenom": string ou null,
  "nom_arabe": string ou null,
  "prenom_arabe": string ou null,
  "date_naissance": string ou null,
  "lieu_naissance": string ou null,
  "sexe": string ou null,
  "nationalite": string ou null,
  "numero_document": string ou null,
  "numero_identification_nationale": string ou null,
  "date_delivrance": string ou null,
  "date_expiration": string ou null,
  "autorite_delivrance": string ou null,
  "adresse": string ou null,
  "langues_detectees": string ou null,
  "champs_incertains": [string],
  "texte_brut_ocr": string
}

Règles impératives :
- Ne traduis JAMAIS les noms propres, adresses ou numéros : recopie-les exactement comme ils apparaissent.
- Si une information est absente ou illisible, mets null plutôt que d'inventer une valeur.
- Liste dans "champs_incertains" le nom de chaque champ que tu n'es pas sûr d'avoir correctement lu.
- "texte_brut_ocr" doit contenir une transcription complète de tout le texte visible, page par page.
"""

print(f"Longueur du prompt d'extraction : {len(EXTRACTION_PROMPT)} caractères.")

In [ ]:
def extract_json_from_text(text: str) -> dict:
    """Extrait le premier objet JSON valide d'une réponse de modèle, même si celle-ci contient des balises
    ```json ... ``` ou du texte superflu avant/après malgré la consigne."""
    cleaned = re.sub(r"^```(?:json)?", "", text.strip()).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    if start == -1:
        raise ValueError("Aucun objet JSON trouvé dans la réponse du modèle.")
    depth = 0
    for i, ch in enumerate(cleaned[start:], start=start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(cleaned[start:i + 1])
    raise ValueError("Objet JSON incomplet dans la réponse du modèle.")

In [ ]:
def process_one_client(client_id: str) -> dict:
    """Traite le JUSTIFICATIF IDENTITE.PDF d'un client : conversion en images, prétraitement, appel au
    modèle, structuration du résultat. Ne lève jamais d'exception : les échecs sont capturés et consignés
    dans le résultat retourné (statut="echec"), pour ne jamais interrompre le traitement par lot."""
    t0 = time.time()
    result = {"client_id": client_id, "statut": "echec", "erreur": None, "nb_pages_traitees": 0}
    pdf_path = FILTERED_DIR / client_id / IDENTITY_FILE

    try:
        if not pdf_path.exists():
            raise FileNotFoundError(f"Fichier introuvable : {pdf_path}")

        pages = pdf_to_images(pdf_path)
        if not pages:
            raise ValueError("Le PDF ne contient aucune page exploitable.")

        processed_pages = [preprocess_page(page_img) for page_img in pages]

        raw_answer = _ask_model(processed_pages, EXTRACTION_PROMPT, MAX_NEW_TOKENS_EXTRACTION)
        if _looks_degenerate(raw_answer):
            raise ValueError(
                f"Réponse du modèle vide ou dégénérée ({raw_answer[:30]!r}...) — symptôme connu d'une "
                "image trop volumineuse envoyée au modèle (voir MAX_IMAGE_DIMENSION_MODEL, Partie 1, et "
                "la checklist de dépannage, Partie 5bis), pas une erreur de format JSON à proprement parler."
            )
        structured = extract_json_from_text(raw_answer)

        result.update(structured)
        result["nb_pages_traitees"] = len(processed_pages)
        result["statut"] = "succes"

    except Exception as e:
        result["erreur"] = str(e)
        logger.error("[%s] échec de l'extraction : %s", client_id, e)

    result["duree_secondes"] = round(time.time() - t0, 1)
    return result

## Partie 7 — Traitement par lot des clients cibles

Chaque client traité produit immédiatement son fichier `RESULTS_DIR/<client_id>.json`. Si le notebook est
interrompu puis relancé, les clients déjà traités sont automatiquement ignorés (sauf `FORCE_REPROCESS =
True` en Partie 1) : le traitement reprend là où il s'était arrêté.

**Sur la vitesse** : la durée de chaque client s'affiche en direct ci-dessous (log + barre de
progression) dès le premier traité — un moyen simple de repérer une lenteur anormale sans attendre la
fin du lot. Si le premier client est déjà beaucoup plus lent qu'attendu, revérifiez d'abord le
diagnostic de répartition GPU/CPU (Partie 5) avant de suspecter autre chose : c'est de loin le facteur
le plus déterminant. Pour aller plus loin (traiter plusieurs clients en un seul appel batché au modèle,
ou chevaucher le prétraitement CPU du client suivant avec l'inférence GPU du client en cours), voir la
note en fin de Partie 8 — non implémenté ici car cela ajoute une vraie complexité (gestion du padding
entre documents de tailles différentes, pression mémoire GPU) qu'il vaut mieux valider dans votre
environnement réel plutôt que de figer une hypothèse non testable ici.

In [ ]:
n_succes = n_echec = 0
progress = tqdm(target_clients, desc="Extraction JUSTIFICATIF IDENTITE")
for client_id in progress:
    out_path = RESULTS_DIR / f"{client_id}.json"
    if out_path.exists() and not FORCE_REPROCESS:
        continue  # déjà traité lors d'une exécution précédente

    result = process_one_client(client_id)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    logger.info("[%s] statut=%s (%ss)", client_id, result["statut"], result["duree_secondes"])

    if result["statut"] == "succes":
        n_succes += 1
    else:
        n_echec += 1
    progress.set_postfix(succes=n_succes, echec=n_echec)

print(f"\nTerminé. {n_succes} succès / {n_echec} échec(s) sur ce passage. "
      f"Résultats individuels disponibles dans {RESULTS_DIR.resolve()}")

## Partie 8 — Consolidation des résultats

On rassemble tous les fichiers JSON individuels en un seul export JSON et un export CSV (pratique pour
Excel / votre équipe conformité), puis on affiche un résumé et la liste des éventuels échecs à examiner
manuellement.

In [ ]:
all_results = []
for f in sorted(RESULTS_DIR.glob("*.json")):
    with open(f, encoding="utf-8") as fh:
        all_results.append(json.load(fh))

if not all_results:
    print("Aucun résultat à consolider pour le moment (le traitement par lot n'a peut-être pas encore été exécuté).")
else:
    with open(COMBINED_RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    results_df = pd.json_normalize(all_results)
    results_df.to_csv(COMBINED_RESULTS_CSV, index=False, encoding="utf-8-sig")

    nb_succes = int((results_df["statut"] == "succes").sum())
    nb_echec = int((results_df["statut"] == "echec").sum())
    print(f"{nb_succes} succès / {nb_echec} échec(s) sur {len(results_df)} client(s) cible(s) traité(s).")

    if nb_echec:
        print("\nClients en échec (à vérifier manuellement) :")
        print(results_df.loc[results_df["statut"] == "echec", ["client_id", "erreur"]].to_string(index=False))

    logger.info("Consolidation terminée -> %s / %s", COMBINED_RESULTS_JSON, COMBINED_RESULTS_CSV)

results_df.head(10) if all_results else None

## Conclusion & prochaines étapes

- Le rapport d'inventaire (`rapport_inventaire_kyc.csv`) couvre les **5** types de documents ; seule
  l'extraction (Parties 4 à 8) se limite pour l'instant à `JUSTIFICATIF IDENTITE.PDF`, comme demandé.
- Pour traiter un autre type de document (`JUSTIFICATIF DOMICILE.PDF`, `FATCA.PDF`...), dupliquez les
  Parties 6 à 8 en adaptant `IDENTITY_FILE` et le schéma JSON du prompt aux champs propres à ce document.
- Avant un passage à l'échelle, validez la qualité de l'extraction sur un échantillon (10-20 clients) en
  comparant manuellement `resultats_extraction_identite.csv` aux documents sources, et affinez le prompt
  si nécessaire — en particulier pour les champs qui reviennent souvent dans `champs_incertains`.
- Vérifiez que ce traitement s'inscrit bien dans votre politique de confidentialité des données / conformité
  RGPD (ou équivalent local), notamment sur la durée de conservation des fichiers intermédiaires générés
  (`kyc_pipeline_workdir/`).
- **Pour aller plus loin sur la vitesse** (Partie 7), au-delà des optimisations déjà en place
  (redimensionnement avant le modèle, `attn_implementation="sdpa"`, diagnostic GPU/CPU) : un vrai gain
  supplémentaire viendrait soit du **traitement par lot batché** (plusieurs clients en un seul appel
  `generate`), soit du **chevauchement** du prétraitement CPU du client suivant avec l'inférence GPU du
  client en cours (pendant que le GPU travaille sur un client, préparer les images du suivant). Les deux
  ajoutent une complexité réelle (le batché demande de gérer un nombre de pages variable par client via
  du padding/attention mask ; le chevauchement demande une file d'attente/un thread dédié) qu'il est
  préférable de valider directement dans votre environnement Domino plutôt que de figer ici une
  hypothèse non testable sans accès à votre GPU réel.